### Using simple Python

In [ ]:
import asyncio
import httpx
from motor.motor_asyncio import AsyncIOMotorClient

* Initialize db

In [ ]:
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "clinical_trials_db"
COLLECTION_NAME = "trials"

* Fetch

In [ ]:
async def fetch_trials():
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {"query.cond": "diabetes", "pageSize": 10}

    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return response.json()

* Parse + Store into the DB

In [ ]:
async def save_trials(data):
    client = AsyncIOMotorClient(MONGO_URI)
    collection = client[DB_NAME][COLLECTION_NAME]

    for study in data["studies"]:
        identification = study["protocolSection"]["identificationModule"]
        nct_id = identification["nctId"]

        await collection.update_one(
            {"_id": nct_id},
            {"$set": study},
            upsert=True,
        )

    print(f"Saved {len(data['studies'])} trials")
    client.close()


In [ ]:
async def main():
    data = await fetch_trials()
    await save_trials(data)


asyncio.run(main())

### Using Fast API

In [ ]:
from contextlib import asynccontextmanager

import httpx
from fastapi import FastAPI
from motor.motor_asyncio import AsyncIOMotorClient

In [ ]:
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "clinical_trials_db"
COLLECTION_NAME = "trials"


* Initialize DB

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    # startup: create the client once, reuse across all requests
    app.mongodb_client = AsyncIOMotorClient(MONGO_URI)
    app.collection = app.mongodb_client[DB_NAME][COLLECTION_NAME]
    yield
    # shutdown: close it cleanly
    app.mongodb_client.close()

In [ ]:
app = FastAPI(lifespan=lifespan)

* Fetch

In [ ]:
async def fetch_trials(condition: str, page_size: int):
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {"query.cond": condition, "pageSize": page_size}

    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return response.json()

* Store

In [ ]:
@app.post("/trials/fetch")
async def fetch_and_save(condition: str = "diabetes", page_size: int = 10):
    data = await fetch_trials(condition, page_size)

    for study in data["studies"]:
        identification = study["protocolSection"]["identificationModule"]
        nct_id = identification["nctId"]

        await app.collection.update_one(
            {"_id": nct_id},
            {"$set": study},
            upsert=True,
        )

    return {"saved": len(data["studies"])}

In [ ]:
@app.get("/trials/{nct_id}")
async def get_trial(nct_id: str):
    trial = await app.collection.find_one({"_id": nct_id})
    if trial is None:
        return {"error": "not found"}
    return trial